# 07. 영향요인 및 오류분석

이 노트북은 3장에 필요한 내용을 만듭니다.

1. 최종 수치예측 `curve_final`이 잘 맞는 조건과 오차가 큰 조건
2. 최종 피크경보 `LightGBM 65% + GRU 35% 앙상블@176`의 미탐지와 오경보 조건
3. 실제 피크가 자주 나타나는 생산조건
4. 수치예측 큰 오차·미탐지·오경보의 대표 사례
5. 최종 경보는 그대로 유지하면서 `164.5 이상 176 미만`을 주의로 표시했을 때의 변화

새 모델은 학습하지 않습니다. 수치예측은 02의 최종 예측을 사용하고, 피크경보는 03-3에서 선정한 LightGBM 65%와 GRU 35% 앙상블 예측을 사용합니다. 같은 실제 시점이 여러 예측창에서 반복되지 않도록 **매일 00시 기준 향후 24시간 예측만** 조건 분석에 사용합니다. 따라서 이 장의 전체 점수는 2장의 모든 예측칸 점수와 다를 수 있습니다.

주의 기준 164.5는 7월에서 재현율 0.95 이상을 만족하도록 정한 뒤 8~9월에 그대로 적용합니다. 최종 경보 기준 176과 기존 성능표는 바꾸지 않습니다.

조건별 차이는 함께 나타난 관계를 보여주며 직접적인 원인을 증명하지 않습니다.


In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from IPython.display import display

start_path = Path.cwd()
root_candidates = [start_path, start_path.parent]
ROOT = next((
    candidate for candidate in root_candidates
    if (candidate / 'dataset/okm_augumented_2021.csv').exists()
    and (candidate / 'results/team_submission_runtime/outputs').exists()
), None)
if ROOT is None:
    raise FileNotFoundError('프로젝트 루트 또는 real_final 폴더에서 실행하세요.')

DATA_PATH = ROOT / 'dataset/okm_augumented_2021.csv'
PRED_PATHS = {
    '7월 검증': ROOT / 'results/team_submission_runtime/outputs/validation_predictions_7월.csv',
    '8~9월 개발 검증': ROOT / 'results/team_submission_runtime/outputs/test_predictions_8_9월.csv',
}
ENSEMBLE_PRED_PATH = (
    ROOT / 'results/ensemble_lgbm_gru_team_aligned/predictions_all_7_9월.csv'
)
OUT = ROOT / 'results/error_peak_condition_analysis_ensemble_fixed176'
FIG = OUT / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

PEAK_LIMIT = 176.0
WARNING_LIMIT = 164.5
SUSPECT_DATES = pd.to_datetime(['2021-07-13', '2021-07-15'])
HOLIDAYS = pd.to_datetime([
    '2021-01-01', '2021-02-11', '2021-02-12', '2021-02-13',
    '2021-03-01', '2021-05-05', '2021-05-19', '2021-06-06',
    '2021-08-15', '2021-08-16', '2021-09-20', '2021-09-21',
    '2021-09-22', '2021-10-03', '2021-10-04', '2021-10-09',
    '2021-10-11', '2021-12-25'
])

font_path = Path(os.environ.get('WINDIR', r'C:\Windows')) / 'Fonts/malgun.ttf'
if font_path.exists():
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['axes.unicode_minus'] = False

if not DATA_PATH.exists():
    raise FileNotFoundError(f'원본 데이터가 없습니다: {DATA_PATH}')
for period, prediction_path in PRED_PATHS.items():
    if not prediction_path.exists():
        raise FileNotFoundError(f'{period} 예측 파일이 없습니다: {prediction_path}')
if not ENSEMBLE_PRED_PATH.exists():
    raise FileNotFoundError(f'앙상블 예측 파일이 없습니다: {ENSEMBLE_PRED_PATH}')

print('원본 데이터:', DATA_PATH.resolve())
print('최종 수치예측:', {k: str(v.resolve()) for k, v in PRED_PATHS.items()})
print('최종 피크경보:', ENSEMBLE_PRED_PATH.resolve())
print('결과 저장:', OUT.resolve())


## 1. 원본 데이터를 15분 단위 조건표로 변환

시간별 한 행에 들어 있는 15·30·45·60분 전력을 15분 단위 행으로 펼칩니다. 각 시점에 가동 여부, 생산량 수준, 생산 시작·종료, 생산량 급증·급감, 주말·공휴일, 1·2주 전 생산상태 일치 여부를 붙입니다.


In [ ]:
raw = pd.read_csv(DATA_PATH)
raw['source_date'] = pd.to_datetime(raw['날짜'].astype(str), format='%Y%m%d')

day_size = raw.groupby('source_date', sort=False).size()
if not day_size.eq(24).all():
    raise ValueError('모든 날짜가 24개 시간 행으로 구성되지 않았습니다.')

raw['row_hour'] = raw.groupby('source_date', sort=False).cumcount()
raw['original_hour'] = pd.to_numeric(raw['시간'], errors='coerce')
raw['time_repaired'] = raw['original_hour'].ne(raw['row_hour'])
raw['hour'] = raw['row_hour']
raw['hour_start'] = raw['source_date'] + pd.to_timedelta(raw['hour'], unit='h')
raw['production'] = pd.to_numeric(raw['생산량'], errors='coerce')

if raw[['production', '15분', '30분', '45분', '60분']].isna().any().any():
    raise ValueError('생산량 또는 전력값에 결측치가 있습니다.')

raw['previous_production'] = raw['production'].shift(1)
raw.loc[raw['source_date'].ne(raw['source_date'].shift(1)), 'previous_production'] = 0.0
raw['production_change'] = raw['production'] - raw['previous_production']

positive_change = raw.loc[raw['production_change'].abs().gt(0), 'production_change'].abs()
RAPID_CHANGE_LIMIT = float(positive_change.quantile(0.75))

is_start = raw['production'].gt(0) & raw['previous_production'].le(0)
is_stop = raw['production'].le(0) & raw['previous_production'].gt(0)
raw['production_transition'] = np.select(
    [is_start, is_stop,
     raw['production_change'].ge(RAPID_CHANGE_LIMIT),
     raw['production_change'].le(-RAPID_CHANGE_LIMIT)],
    ['생산 시작', '생산 종료', '생산량 급증', '생산량 급감'],
    default='큰 변화 없음'
)

positive_production = raw.loc[raw['production'].gt(0), 'production']
PROD_Q1, PROD_Q2 = positive_production.quantile([1/3, 2/3]).astype(float)
raw['production_level'] = np.select(
    [raw['production'].le(0), raw['production'].le(PROD_Q1), raw['production'].le(PROD_Q2)],
    ['비생산', '낮은 생산량', '중간 생산량'],
    default='높은 생산량'
)

long_parts = []
for minute, power_col in zip([15, 30, 45, 60], ['15분', '30분', '45분', '60분']):
    part = raw[[
        'source_date', 'hour_start', 'production', 'previous_production',
        'production_change', 'production_transition', 'production_level',
        'time_repaired', '전기요금(계절)', '인건비'
    ]].copy()
    part['target_time'] = part['hour_start'] + pd.to_timedelta(minute, unit='m')
    part['actual_power_raw'] = pd.to_numeric(raw[power_col], errors='coerce')
    part['quarter_minute'] = minute
    long_parts.append(part)

actual = pd.concat(long_parts, ignore_index=True).sort_values('target_time').reset_index(drop=True)
if actual['target_time'].duplicated().any():
    raise ValueError('15분 단위 실제 시각이 중복되었습니다.')

actual['calendar_date'] = actual['target_time'].dt.normalize()
actual['month'] = actual['target_time'].dt.strftime('%Y-%m')
actual['hour'] = actual['target_time'].dt.hour
actual['time_band'] = pd.cut(
    actual['hour'], bins=[-1, 5, 11, 17, 23],
    labels=['새벽(0~5시)', '오전(6~11시)', '오후(12~17시)', '저녁(18~23시)']
).astype(str)
actual['weekend'] = actual['calendar_date'].dt.dayofweek.ge(5)
actual['holiday'] = actual['calendar_date'].isin(HOLIDAYS)
actual['day_type'] = np.select(
    [actual['holiday'], actual['weekend']], ['공휴일', '주말'], default='평일'
)

state_by_time = actual.set_index('target_time')['production'].gt(0)
state_now = actual['production'].gt(0).to_numpy()
state_w1 = state_by_time.reindex(actual['target_time'] - pd.Timedelta(days=7)).to_numpy()
state_w2 = state_by_time.reindex(actual['target_time'] - pd.Timedelta(days=14)).to_numpy()
valid_w1 = pd.notna(state_w1)
valid_w2 = pd.notna(state_w2)
mismatch_w1 = valid_w1 & (state_now != state_w1)
mismatch_w2 = valid_w2 & (state_now != state_w2)
actual['weekly_state_match'] = np.select(
    [~valid_w1 | ~valid_w2,
     mismatch_w1 & mismatch_w2,
     mismatch_w1,
     mismatch_w2],
    ['과거 부족', '1·2주 모두 불일치', '1주만 불일치', '2주만 불일치'],
    default='1·2주 모두 일치'
)
actual['operation_state'] = np.where(actual['production'].gt(0), '가동 중', '비가동')
actual['power_level'] = pd.cut(
    actual['actual_power_raw'],
    bins=[-np.inf, 100, 150, PEAK_LIMIT, 200, np.inf],
    right=False,
    labels=['낮음(100 미만)', '보통(100~149)', '피크 근처(150~175)',
            '피크(176~199)', '큰 피크(200 이상)']
).astype(str)
actual['actual_peak'] = actual['actual_power_raw'].ge(PEAK_LIMIT)
actual['peak_excess'] = (actual['actual_power_raw'] - PEAK_LIMIT).clip(lower=0)
actual['suspect_date'] = actual['source_date'].isin(SUSPECT_DATES)

print(f'원본 시간 행: {len(raw):,} | 15분 관측: {len(actual):,}')
print('시간 복구 날짜:', raw.loc[raw['time_repaired'], 'source_date'].dt.strftime('%Y-%m-%d').unique().tolist())
print(f'생산량 구간 경계: {PROD_Q1:.3f}, {PROD_Q2:.3f}')
print(f'급격한 생산량 변화 기준: |변화량| >= {RAPID_CHANGE_LIMIT:.3f}')


## 2. 최종 예측과 실제 조건 연결

수치예측은 `curve_final`을 사용합니다. 피크경보는 LightGBM 개선 Q1(q=.75) 65%와 GRU 개선 입력(q=.80) 35%를 결합한 `ensemble` 값이 176 이상인지로 판단합니다. 7월과 8~9월 예측을 합친 뒤 매일 00시에 생성된 24시간 예측만 남겨 실제 시점의 중복을 제거합니다.

운영 단계는 다음과 같이 구분합니다.

- 경보: 앙상블 예측값 176 이상
- 주의: 앙상블 예측값 164.5 이상 176 미만
- 정상: 앙상블 예측값 164.5 미만

주의 단계는 최종 경보 기준을 바꾸는 것이 아니라, 경보 직전 구간을 미리 확인하기 위한 추가 운영 규칙입니다. 기존 오류분석은 경보 기준 176으로 계산합니다.


In [ ]:
required_columns = [
    '기준시각', '예측대상시각', '예측칸', '실제전력', 'curve_final'
]
pred_parts = []
for period, prediction_path in PRED_PATHS.items():
    part = pd.read_csv(prediction_path, usecols=required_columns)
    part.insert(0, 'evaluation_period', period)
    pred_parts.append(part)
pred = pd.concat(pred_parts, ignore_index=True)

prediction_keys = ['기준시각', '예측대상시각', '예측칸']
peak_required_columns = prediction_keys + [
    'actual', 'lgbm_q075', 'gru_q080', 'ensemble', 'threshold'
]
peak_pred = pd.read_csv(ENSEMBLE_PRED_PATH, usecols=peak_required_columns)
if peak_pred.duplicated(prediction_keys).any():
    raise ValueError('앙상블 예측 파일의 예측 키가 중복되었습니다.')
pred = pred.merge(peak_pred, on=prediction_keys, how='left', validate='one_to_one')
if pred['ensemble'].isna().any():
    raise ValueError('일부 수치예측과 앙상블 예측을 연결하지 못했습니다.')

ensemble_difference = (
    pred['ensemble'] - (0.65 * pred['lgbm_q075'] + 0.35 * pred['gru_q080'])
).abs().max()
if ensemble_difference > 1e-6:
    raise ValueError(f'앙상블 비율이 65:35와 다릅니다. 최대 차이={ensemble_difference}')
if not pd.to_numeric(pred['threshold'], errors='coerce').eq(PEAK_LIMIT).all():
    raise ValueError('앙상블 예측 파일의 경보 기준이 176과 다릅니다.')

pred['origin'] = pd.to_datetime(pred['기준시각'])
pred['target_time'] = pd.to_datetime(pred['예측대상시각'])
daily_pred = pred.loc[
    pred['origin'].dt.hour.eq(0) & pred['origin'].dt.minute.eq(0)
].copy()

if daily_pred['target_time'].duplicated().any():
    duplicated = daily_pred.loc[
        daily_pred['target_time'].duplicated(False), 'target_time'
    ].head()
    raise ValueError(f'일별 예측에서 대상시각이 중복되었습니다: {duplicated.tolist()}')

analysis = daily_pred.merge(actual, on='target_time', how='left', validate='one_to_one')
if analysis['actual_power_raw'].isna().any():
    raise ValueError('일부 예측시각을 원본 15분 데이터와 연결하지 못했습니다.')

actual_difference = (
    pd.to_numeric(analysis['실제전력'], errors='coerce')
    - analysis['actual_power_raw']
).abs().max()
if actual_difference > 1e-6:
    raise ValueError(
        f'예측 파일과 원본의 실제 전력값이 다릅니다. 최대 차이={actual_difference}'
    )
ensemble_actual_difference = (
    pd.to_numeric(analysis['actual'], errors='coerce')
    - analysis['actual_power_raw']
).abs().max()
if ensemble_actual_difference > 1e-6:
    raise ValueError(
        f'앙상블 파일과 원본의 실제 전력값이 다릅니다. 최대 차이={ensemble_actual_difference}'
    )

analysis['actual_power'] = pd.to_numeric(analysis['실제전력'], errors='coerce')
analysis['point_prediction'] = pd.to_numeric(analysis['curve_final'], errors='coerce')
analysis['lgbm_peak_prediction'] = pd.to_numeric(analysis['lgbm_q075'], errors='coerce')
analysis['gru_peak_prediction'] = pd.to_numeric(analysis['gru_q080'], errors='coerce')
analysis['peak_prediction'] = pd.to_numeric(analysis['ensemble'], errors='coerce')
analysis['point_error'] = analysis['point_prediction'] - analysis['actual_power']
analysis['absolute_error'] = analysis['point_error'].abs()
analysis['squared_error'] = analysis['point_error'].pow(2)
analysis['lead_hour'] = (
    analysis['target_time'] - analysis['origin']
).dt.total_seconds() / 3600
analysis['horizon_band'] = pd.cut(
    analysis['lead_hour'], bins=[0, 6, 12, 18, 24.001], include_lowest=True,
    labels=['0~6시간', '6~12시간', '12~18시간', '18~24시간']
).astype(str)
analysis['predicted_alarm'] = analysis['peak_prediction'].ge(PEAK_LIMIT)
analysis['predicted_attention'] = analysis['peak_prediction'].ge(WARNING_LIMIT)
analysis['operation_level'] = np.select(
    [analysis['predicted_alarm'], analysis['predicted_attention']],
    ['경보', '주의'], default='정상'
)
analysis['actual_peak'] = analysis['actual_power'].ge(PEAK_LIMIT)
analysis['alarm_result'] = np.select(
    [
        analysis['actual_peak'] & analysis['predicted_alarm'],
        analysis['actual_peak'] & ~analysis['predicted_alarm'],
        ~analysis['actual_peak'] & analysis['predicted_alarm'],
    ],
    ['정확한 경보', '미탐지', '오경보'],
    default='정확한 정상 판단',
)

LARGE_ERROR_LIMIT = float(analysis['absolute_error'].quantile(0.90))
analysis['large_error'] = analysis['absolute_error'].ge(LARGE_ERROR_LIMIT)

print(f'전체 롤링 예측: {len(pred):,}칸')
print(f'일별 00시 기준 평가: {analysis["origin"].nunique():,}일, {len(analysis):,}칸')
print(f'평가기간: {analysis["target_time"].min()} ~ {analysis["target_time"].max()}')
print(f'전체 MAE: {analysis["absolute_error"].mean():.3f}')
print(f'큰 오차 기준(상위 10%): {LARGE_ERROR_LIMIT:.3f}')
print(f'실제 피크 비율: {analysis["actual_peak"].mean():.2%}')
display(analysis['alarm_result'].value_counts().rename_axis('판정').to_frame('건수'))


## 3. 수치예측이 잘 맞는 조건과 오차가 큰 조건

각 조건에서 표본 수, MAE, RMSE, 실제보다 높거나 낮게 예측한 정도, 큰 오차 비율을 계산합니다. `큰 오차`는 절대오차가 상위 10%인 경우입니다.

표본 수가 적은 조건은 우연에 따라 결과가 크게 달라질 수 있으므로 표본 수와 함께 해석합니다.


In [ ]:
def error_summary(frame, condition):
    rows = []
    for value, group in frame.groupby(condition, observed=True, dropna=False):
        rows.append({
            '조건값': str(value),
            '표본수': len(group),
            'MAE': group['absolute_error'].mean(),
            'RMSE': np.sqrt(group['squared_error'].mean()),
            '평균오차_예측-실제': group['point_error'].mean(),
            '큰오차수': int(group['large_error'].sum()),
            '큰오차비율': group['large_error'].mean(),
            '실제평균전력': group['actual_power'].mean(),
        })
    return pd.DataFrame(rows)

error_conditions = {
    '평가기간': 'evaluation_period',
    '월': 'month',
    '시간대': 'time_band',
    '요일유형': 'day_type',
    '가동상태': 'operation_state',
    '생산량수준': 'production_level',
    '생산변화': 'production_transition',
    '실제전력수준': 'power_level',
    '주간생산상태': 'weekly_state_match',
    '예측거리': 'horizon_band',
}

error_tables = {}
for name, column in error_conditions.items():
    table = error_summary(analysis, column)
    error_tables[name] = table
    table.to_csv(OUT / f'error_by_{column}.csv', index=False, encoding='utf-8-sig')

top_errors = analysis.nlargest(30, 'absolute_error')[[
    'evaluation_period', 'origin', 'target_time', 'actual_power',
    'point_prediction', 'point_error', 'absolute_error', 'production',
    'production_change', 'production_transition', 'production_level',
    'operation_state', 'power_level', 'day_type', 'weekly_state_match',
    'horizon_band'
]]
top_errors.to_csv(OUT / 'top_error_cases.csv', index=False, encoding='utf-8-sig')

for name, table in error_tables.items():
    print()
    print(f'[{name}별 수치예측 오차]')
    display(table.round(3))


In [ ]:
plot_names = ['평가기간', '시간대', '요일유형', '가동상태', '생산변화', '실제전력수준']
fig, axes = plt.subplots(3, 2, figsize=(15, 14))

for ax, name in zip(axes.flat, plot_names):
    table = error_tables[name]
    x = np.arange(len(table))
    bars = ax.bar(x, table['MAE'], color='#3B82F6')
    ax.axhline(
        analysis['absolute_error'].mean(), color='#DC2626', linestyle='--',
        label=f'전체 MAE {analysis["absolute_error"].mean():.2f}'
    )
    ax.set_title(f'{name}별 수치예측 MAE')
    ax.set_ylabel('MAE')
    ax.set_xticks(x, table['조건값'], rotation=25, ha='right')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(fontsize=8)
    for bar, count in zip(bars, table['표본수']):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height(), f'n={count:,}',
            ha='center', va='bottom', fontsize=7
        )

fig.suptitle('어떤 조건에서 전력 수치예측 오차가 커지는가', fontsize=16, y=1.01)
plt.tight_layout()
fig.savefig(FIG / 'error_conditions.png', dpi=160, bbox_inches='tight')
plt.show()


## 4. 실제 피크가 많이 발생하는 조건

피크 기준은 176입니다. 실제 피크 조건은 원본 전체 기간에서 분석하되, 시간 기록을 복구한 7월 13일과 15일은 제외합니다. 조건별 피크 발생률과 기준 초과량을 함께 확인합니다.


In [ ]:
peak_data = actual.loc[~actual['suspect_date']].copy()

def peak_summary(frame, condition):
    rows = []
    for value, group in frame.groupby(condition, observed=True, dropna=False):
        peak_group = group.loc[group['actual_peak']]
        rows.append({
            '조건값': str(value),
            '표본수': len(group),
            '피크발생수': int(group['actual_peak'].sum()),
            '피크발생률': group['actual_peak'].mean(),
            '평균전력': group['actual_power_raw'].mean(),
            '최대전력': group['actual_power_raw'].max(),
            '총피크초과량': group['peak_excess'].sum(),
            '피크시평균초과량': peak_group['peak_excess'].mean() if len(peak_group) else 0.0,
        })
    return pd.DataFrame(rows)

peak_conditions = {
    '월': 'month',
    '시간': 'hour',
    '시간대': 'time_band',
    '요일유형': 'day_type',
    '가동상태': 'operation_state',
    '생산량수준': 'production_level',
    '생산변화': 'production_transition',
    '주간생산상태': 'weekly_state_match',
}

peak_tables = {}
for name, column in peak_conditions.items():
    table = peak_summary(peak_data, column)
    peak_tables[name] = table
    table.to_csv(OUT / f'peak_by_{column}.csv', index=False, encoding='utf-8-sig')

ordered_peak = peak_data.sort_values('target_time').copy()
previous_peak = ordered_peak['actual_peak'].shift(fill_value=False)
time_gap = ordered_peak['target_time'].diff().ne(pd.Timedelta(minutes=15))
event_start = ordered_peak['actual_peak'] & (~previous_peak | time_gap)
ordered_peak['peak_event_id'] = event_start.cumsum()
peak_events = (
    ordered_peak.loc[ordered_peak['actual_peak']]
    .groupby('peak_event_id')
    .agg(
        시작시각=('target_time', 'min'), 종료시각=('target_time', 'max'),
        지속칸=('target_time', 'size'), 최대전력=('actual_power_raw', 'max'),
        총초과량=('peak_excess', 'sum')
    ).reset_index(drop=True)
)
peak_events['지속시간_분'] = peak_events['지속칸'] * 15
peak_events.to_csv(OUT / 'peak_events.csv', index=False, encoding='utf-8-sig')

top_peaks = peak_data.nlargest(30, 'actual_power_raw')[[
    'target_time', 'actual_power_raw', 'peak_excess', 'production',
    'production_change', 'production_transition', 'production_level',
    'day_type', 'weekly_state_match'
]]
top_peaks.to_csv(OUT / 'top_peak_cases.csv', index=False, encoding='utf-8-sig')

print(f'분석 관측치: {len(peak_data):,}개')
print(f'피크 관측치: {peak_data["actual_peak"].sum():,}개 ({peak_data["actual_peak"].mean():.2%})')
print(f'연속 피크 사건: {len(peak_events):,}건')
display(peak_events['지속시간_분'].describe().to_frame('피크 지속시간(분)').round(2))

for name in ['월', '시간대', '요일유형', '가동상태', '생산량수준', '생산변화', '주간생산상태']:
    print()
    print(f'[{name}별 실제 피크]')
    display(peak_tables[name].round(3))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

hour_table = peak_tables['시간'].sort_values('조건값', key=lambda s: pd.to_numeric(s))
axes[0, 0].plot(pd.to_numeric(hour_table['조건값']), hour_table['피크발생률'] * 100,
                marker='o', color='#DC2626')
axes[0, 0].set_title('시간별 실제 피크 발생률')
axes[0, 0].set_xlabel('시간')
axes[0, 0].set_ylabel('피크 발생률(%)')
axes[0, 0].set_xticks(range(0, 24, 2))

for ax, name in zip(axes.flat[1:], ['요일유형', '생산량수준', '생산변화']):
    table = peak_tables[name]
    x = np.arange(len(table))
    bars = ax.bar(x, table['피크발생률'] * 100, color='#F97316')
    ax.set_title(f'{name}별 실제 피크 발생률')
    ax.set_ylabel('피크 발생률(%)')
    ax.set_xticks(x, table['조건값'], rotation=25, ha='right')
    for bar, count in zip(bars, table['표본수']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'n={count:,}',
                ha='center', va='bottom', fontsize=7)

for ax in axes.flat:
    ax.grid(axis='y', alpha=0.25)

fig.suptitle('최대피크는 어떤 조건에서 발생하는가', fontsize=16, y=1.01)
plt.tight_layout()
fig.savefig(FIG / 'peak_conditions.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. 미탐지와 오경보가 발생하는 조건

최종 피크경보가 어느 조건에서 미탐지와 오경보를 만드는지 확인합니다. 정밀도·재현율·F1과 함께 미탐지율과 오경보율을 계산합니다.

- 미탐지율: 실제 피크 중 경보하지 못한 비율
- 오경보율: 실제 정상 중 잘못 경보한 비율


In [ ]:
def alarm_summary(frame, condition):
    rows = []
    for value, group in frame.groupby(condition, observed=True, dropna=False):
        actual_positive = group['actual_peak'].to_numpy(bool)
        predicted_positive = group['predicted_alarm'].to_numpy(bool)
        tp = int(np.sum(actual_positive & predicted_positive))
        fp = int(np.sum(~actual_positive & predicted_positive))
        fn = int(np.sum(actual_positive & ~predicted_positive))
        tn = int(np.sum(~actual_positive & ~predicted_positive))
        precision = tp / (tp + fp) if tp + fp else np.nan
        recall = tp / (tp + fn) if tp + fn else np.nan
        f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan
        rows.append({
            '조건값': str(value), '표본수': len(group),
            '실제피크수': tp + fn, '예측경보수': tp + fp,
            'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            '정밀도': precision, '재현율': recall, 'F1': f1,
            '미탐지율': fn / (tp + fn) if tp + fn else np.nan,
            '오경보율': fp / (fp + tn) if fp + tn else np.nan,
        })
    return pd.DataFrame(rows)

alarm_conditions = {
    '평가기간': 'evaluation_period',
    '월': 'month',
    '시간대': 'time_band',
    '요일유형': 'day_type',
    '가동상태': 'operation_state',
    '생산량수준': 'production_level',
    '생산변화': 'production_transition',
    '실제전력수준': 'power_level',
    '주간생산상태': 'weekly_state_match',
    '예측거리': 'horizon_band',
}

alarm_tables = {}
for name, column in alarm_conditions.items():
    table = alarm_summary(analysis, column)
    alarm_tables[name] = table
    table.to_csv(OUT / f'alarm_by_{column}.csv', index=False, encoding='utf-8-sig')

case_columns = [
    'evaluation_period', 'origin', 'target_time', 'actual_power',
    'point_prediction', 'lgbm_peak_prediction', 'gru_peak_prediction',
    'peak_prediction', 'alarm_result', 'production',
    'production_change', 'production_transition', 'production_level',
    'operation_state', 'power_level', 'day_type', 'weekly_state_match',
    'horizon_band'
]
false_alarms = analysis.loc[analysis['alarm_result'].eq('오경보'), case_columns].copy()
missed_peaks = analysis.loc[analysis['alarm_result'].eq('미탐지'), case_columns].copy()
false_alarms['경보기준초과량'] = false_alarms['peak_prediction'] - PEAK_LIMIT
missed_peaks['실제피크초과량'] = missed_peaks['actual_power'] - PEAK_LIMIT
false_alarms.sort_values('경보기준초과량', ascending=False).to_csv(
    OUT / 'false_alarm_cases.csv', index=False, encoding='utf-8-sig'
)
missed_peaks.sort_values('실제피크초과량', ascending=False).to_csv(
    OUT / 'missed_peak_cases.csv', index=False, encoding='utf-8-sig'
)

for name, table in alarm_tables.items():
    print()
    print(f'[{name}별 피크경보 성능]')
    display(table.round(3))


## 5.1 주의 단계 추가 운영 분석

최종 경보는 앙상블 예측값 176 이상으로 유지합니다. 다만 164.5 이상 176 미만을 `주의`로 표시하여 설비 가동 계획을 미리 확인하도록 합니다.

주의 기준 164.5는 7월 데이터에서 재현율 0.95 이상을 만족하는 값으로 정했습니다. 이후 기준을 바꾸지 않고 8~9월에 적용했습니다. 8~9월 비교는 제2장과 같은 모든 24시간 예측창을 사용하며, 조건별 오류분석에 사용하는 매일 00시 예측 결과도 별도 파일로 저장합니다.


In [ ]:
def binary_metrics(frame, prediction_column):
    actual_positive = frame['actual_peak'].to_numpy(bool)
    predicted_positive = frame[prediction_column].to_numpy(bool)
    tp = int(np.sum(actual_positive & predicted_positive))
    fp = int(np.sum(~actual_positive & predicted_positive))
    fn = int(np.sum(actual_positive & ~predicted_positive))
    tn = int(np.sum(~actual_positive & ~predicted_positive))
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan
    return {
        '표본수': len(frame), 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        '정밀도': precision, '재현율': recall, 'F1': f1,
        '미탐지율': fn / (tp + fn) if tp + fn else np.nan,
    }


rolling_policy = pred.copy()
rolling_policy['actual_power'] = pd.to_numeric(rolling_policy['actual'], errors='coerce')
rolling_policy['peak_prediction'] = pd.to_numeric(rolling_policy['ensemble'], errors='coerce')
rolling_policy['actual_peak'] = rolling_policy['actual_power'].ge(PEAK_LIMIT)
rolling_policy['alarm_only'] = rolling_policy['peak_prediction'].ge(PEAK_LIMIT)
rolling_policy['alarm_or_warning'] = rolling_policy['peak_prediction'].ge(WARNING_LIMIT)
rolling_policy['operation_level'] = np.select(
    [rolling_policy['alarm_only'], rolling_policy['alarm_or_warning']],
    ['경보', '주의'], default='정상'
)

july = rolling_policy.loc[rolling_policy['evaluation_period'].eq('7월 검증')]
aug_sep = rolling_policy.loc[rolling_policy['evaluation_period'].eq('8~9월 개발 검증')]

july_warning_check = pd.DataFrame([{
    '주의기준': WARNING_LIMIT,
    **binary_metrics(july, 'alarm_or_warning'),
}])
july_warning_check.to_csv(
    OUT / 'warning_threshold_july_check.csv', index=False, encoding='utf-8-sig'
)

warning_policy_full = pd.DataFrame([
    {'운영방식': '경보만(176 이상)', **binary_metrics(aug_sep, 'alarm_only')},
    {'운영방식': '경보+주의(164.5 이상)', **binary_metrics(aug_sep, 'alarm_or_warning')},
])
warning_policy_full.to_csv(
    OUT / 'warning_policy_8_9_full_windows.csv', index=False, encoding='utf-8-sig'
)

daily_policy = pd.DataFrame([
    {'평가기간': '7~9월', '운영방식': '경보만(176 이상)', **binary_metrics(analysis, 'predicted_alarm')},
    {'평가기간': '7~9월', '운영방식': '경보+주의(164.5 이상)', **binary_metrics(analysis, 'predicted_attention')},
    {'평가기간': '8~9월', '운영방식': '경보만(176 이상)', **binary_metrics(
        analysis.loc[analysis['evaluation_period'].eq('8~9월 개발 검증')], 'predicted_alarm'
    )},
    {'평가기간': '8~9월', '운영방식': '경보+주의(164.5 이상)', **binary_metrics(
        analysis.loc[analysis['evaluation_period'].eq('8~9월 개발 검증')], 'predicted_attention'
    )},
])
daily_policy.to_csv(
    OUT / 'warning_policy_daily_00.csv', index=False, encoding='utf-8-sig'
)

return_week = aug_sep.loc[
    aug_sep['target_time'].between('2021-08-09', '2021-08-13 23:59:59')
]
return_week_policy = pd.DataFrame([
    {'운영방식': '경보만(176 이상)', **binary_metrics(return_week, 'alarm_only')},
    {'운영방식': '경보+주의(164.5 이상)', **binary_metrics(return_week, 'alarm_or_warning')},
])
return_week_policy.to_csv(
    OUT / 'warning_policy_return_week_full_windows.csv', index=False, encoding='utf-8-sig'
)

print('[7월 주의 기준 확인: 재현율 0.95 이상]')
display(july_warning_check.round(4))
print('[8~9월 모든 예측창: 경보만 / 경보+주의]')
display(warning_policy_full.round(4))
print('[매일 00시 예측: 경보만 / 경보+주의]')
display(daily_policy.round(4))
print('[8월 9~13일 복귀 주, 모든 예측창]')
display(return_week_policy.round(4))


## 6. 대표 실패 사례

수치예측 오차가 가장 큰 시점, 실제 피크가 가장 큰 미탐지, 예측값이 가장 높은 오경보를 대표 사례로 선택합니다. 각 사례 전후 3시간의 실제 전력, 수치예측, 최종 앙상블 경보 예측과 기준 176을 함께 표시합니다.


In [ ]:
representative_rows = []

if len(analysis):
    row = analysis.loc[analysis['absolute_error'].idxmax()]
    representative_rows.append(('가장 큰 수치예측 오차', row))
if len(missed_peaks):
    row = analysis.loc[missed_peaks['actual_power'].idxmax()]
    representative_rows.append(('가장 큰 실제 피크 미탐지', row))
if len(false_alarms):
    row = analysis.loc[false_alarms['peak_prediction'].idxmax()]
    representative_rows.append(('가장 높은 예측값의 오경보', row))

representative_cases = pd.DataFrame([
    {
        '사례': label,
        '평가기간': row['evaluation_period'],
        '기준시각': row['origin'],
        '예측대상시각': row['target_time'],
        '실제전력': row['actual_power'],
        '수치예측': row['point_prediction'],
        'LightGBM경보예측': row['lgbm_peak_prediction'],
        'GRU경보예측': row['gru_peak_prediction'],
        '앙상블경보예측': row['peak_prediction'],
        '경보결과': row['alarm_result'],
        '가동상태': row['operation_state'],
        '생산변화': row['production_transition'],
    }
    for label, row in representative_rows
])
representative_cases.to_csv(
    OUT / 'representative_error_cases.csv', index=False, encoding='utf-8-sig'
)
display(representative_cases.round(3))

if representative_rows:
    fig, axes = plt.subplots(
        len(representative_rows), 1,
        figsize=(14, 4.2 * len(representative_rows)),
        squeeze=False,
    )
    for ax, (label, row) in zip(axes.flat, representative_rows):
        center = row['target_time']
        window = analysis.loc[
            analysis['target_time'].between(
                center - pd.Timedelta(hours=3),
                center + pd.Timedelta(hours=3),
            )
        ].sort_values('target_time')
        ax.plot(window['target_time'], window['actual_power'], label='실제 전력', linewidth=2)
        ax.plot(window['target_time'], window['point_prediction'], label='수치예측', linewidth=1.5)
        ax.plot(window['target_time'], window['peak_prediction'], label='앙상블 경보 예측', linewidth=1.2)
        ax.axhline(PEAK_LIMIT, color='#DC2626', linestyle='--', label='피크 기준 176')
        ax.axvline(center, color='black', linestyle=':', label='대표 오류 시점')
        ax.set_title(
            f'{label}: {center} | 실제 {row["actual_power"]:.1f}, '
            f'수치예측 {row["point_prediction"]:.1f}, '
            f'앙상블 경보 예측 {row["peak_prediction"]:.1f}'
        )
        ax.set_ylabel('전력')
        ax.grid(alpha=0.25)
        ax.legend(ncol=5, fontsize=8)
    axes.flat[-1].set_xlabel('예측 대상 시각')
    fig.tight_layout()
    fig.savefig(FIG / 'representative_error_cases.png', dpi=160, bbox_inches='tight')
    plt.show()


## 7. 잘 작동하는 조건과 어려운 조건 요약

표본이 너무 적은 조건을 제외하고, 수치예측 MAE가 가장 낮고 높은 조건을 찾습니다. 피크경보는 실제 피크가 10건 이상 있는 조건에서 F1이 가장 높고 낮은 조건을 찾습니다. 보고서에서는 “취약조건” 대신 “오차가 큰 조건”, “미탐지가 많은 조건”처럼 뜻이 바로 드러나는 표현을 사용합니다.


In [ ]:
finding_rows = []
for category, table in error_tables.items():
    eligible = table.loc[table['표본수'].ge(50)].copy()
    if len(eligible):
        best = eligible.loc[eligible['MAE'].idxmin()]
        worst = eligible.loc[eligible['MAE'].idxmax()]
        finding_rows.extend([
            {
                '분석': '수치예측', '분류': category, '구분': '오차가 작은 조건',
                '조건': best['조건값'], '표본수': int(best['표본수']),
                '지표': 'MAE', '값': float(best['MAE']),
            },
            {
                '분석': '수치예측', '분류': category, '구분': '오차가 큰 조건',
                '조건': worst['조건값'], '표본수': int(worst['표본수']),
                '지표': 'MAE', '값': float(worst['MAE']),
            },
        ])

for category, table in alarm_tables.items():
    eligible = table.loc[
        table['표본수'].ge(50) & table['실제피크수'].ge(10) & table['F1'].notna()
    ].copy()
    if len(eligible):
        best = eligible.loc[eligible['F1'].idxmax()]
        worst = eligible.loc[eligible['F1'].idxmin()]
        finding_rows.extend([
            {
                '분석': '피크경보', '분류': category, '구분': '경보가 잘 맞는 조건',
                '조건': best['조건값'], '표본수': int(best['표본수']),
                '지표': 'F1', '값': float(best['F1']),
            },
            {
                '분석': '피크경보', '분류': category, '구분': '경보가 잘 맞지 않는 조건',
                '조건': worst['조건값'], '표본수': int(worst['표본수']),
                '지표': 'F1', '값': float(worst['F1']),
            },
        ])

for category, table in peak_tables.items():
    eligible = table.loc[table['표본수'].ge(100)].copy()
    if len(eligible):
        highest = eligible.loc[eligible['피크발생률'].idxmax()]
        finding_rows.append({
            '분석': '실제피크', '분류': category, '구분': '피크가 자주 발생한 조건',
            '조건': highest['조건값'], '표본수': int(highest['표본수']),
            '지표': '피크발생률', '값': float(highest['피크발생률']),
        })

key_findings = pd.DataFrame(finding_rows)
key_findings.to_csv(OUT / 'chapter3_key_findings.csv', index=False, encoding='utf-8-sig')

overall_tp = int((analysis['actual_peak'] & analysis['predicted_alarm']).sum())
overall_fp = int((~analysis['actual_peak'] & analysis['predicted_alarm']).sum())
overall_fn = int((analysis['actual_peak'] & ~analysis['predicted_alarm']).sum())
overall_tn = int((~analysis['actual_peak'] & ~analysis['predicted_alarm']).sum())
overall_precision = overall_tp / (overall_tp + overall_fp) if overall_tp + overall_fp else np.nan
overall_recall = overall_tp / (overall_tp + overall_fn) if overall_tp + overall_fn else np.nan
overall_f1 = (
    2 * overall_precision * overall_recall / (overall_precision + overall_recall)
    if overall_precision + overall_recall else np.nan
)

summary = {
    'evaluation_rule': '7~9월 중 매일 00시 기준 향후 24시간 예측만 사용',
    'evaluation_start': str(analysis['target_time'].min()),
    'evaluation_end': str(analysis['target_time'].max()),
    'evaluation_days': int(analysis['origin'].nunique()),
    'evaluation_cells': int(len(analysis)),
    'point_model': 'curve_final',
    'peak_model': '0.65 * LightGBM improved_Q1_q075 + 0.35 * GRU enhanced47_q080',
    'peak_lgbm_weight': 0.65,
    'peak_gru_weight': 0.35,
    'peak_limit': float(PEAK_LIMIT),
    'warning_limit': float(WARNING_LIMIT),
    'warning_rule': '7월에서 재현율 0.95 이상을 만족하도록 정한 뒤 8~9월에 고정 적용',
    'point_mae': float(analysis['absolute_error'].mean()),
    'point_rmse': float(np.sqrt(analysis['squared_error'].mean())),
    'point_bias': float(analysis['point_error'].mean()),
    'large_error_p90': float(LARGE_ERROR_LIMIT),
    'alarm_precision': float(overall_precision),
    'alarm_recall': float(overall_recall),
    'alarm_f1': float(overall_f1),
    'true_positive': overall_tp,
    'false_alarm': overall_fp,
    'missed_peak': overall_fn,
    'true_negative': overall_tn,
    'raw_peak_cells_excluding_suspect_dates': int(peak_data['actual_peak'].sum()),
    'raw_peak_rate_excluding_suspect_dates': float(peak_data['actual_peak'].mean()),
    'peak_event_count': int(len(peak_events)),
    'production_low_boundary': float(PROD_Q1),
    'production_high_boundary': float(PROD_Q2),
    'rapid_change_limit': float(RAPID_CHANGE_LIMIT),
    'independent_test_remaining': False,
    'limitations': [
        '조건별 차이는 함께 나타난 관계이며 직접적인 원인을 증명하지 않는다.',
        '7~9월은 모델 선택에 사용했으므로 독립 테스트가 아닌 개발 검증 결과다.',
        '설비별 상태와 제품 종류가 없어 해당 조건은 분석하지 못한다.',
        '중복 제거를 위해 매일 00시 예측만 사용하므로 2장의 전체 예측칸 점수와 다를 수 있다.',
    ],
}
(OUT / 'analysis_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
pd.DataFrame([{
    '평가시작': summary['evaluation_start'],
    '평가종료': summary['evaluation_end'],
    '평가일수': summary['evaluation_days'],
    '평가칸수': summary['evaluation_cells'],
    '수치예측_MAE': summary['point_mae'],
    '수치예측_RMSE': summary['point_rmse'],
    '평균오차_예측-실제': summary['point_bias'],
    '정밀도': summary['alarm_precision'],
    '재현율': summary['alarm_recall'],
    'F1': summary['alarm_f1'],
    '오경보': summary['false_alarm'],
    '미탐지': summary['missed_peak'],
}]).to_csv(OUT / 'chapter3_overall_summary.csv', index=False, encoding='utf-8-sig')

print('[조건별 핵심 결과]')
display(key_findings.round(4))
print()
print('[3장 전체 평가 요약]')
display(pd.DataFrame([summary]).drop(columns='limitations').round(4))
print()
print('저장 완료:', OUT.resolve())


## 8. 3장 해석 원칙과 다음 장 연결

- MAE가 높은 조건은 모델이 그 조건에서 상대적으로 큰 오차를 냈다는 뜻입니다.
- 피크 발생률이 높은 조건은 피크와 함께 자주 관찰된 조건입니다.
- 미탐지와 오경보는 건수와 비율을 함께 제시합니다.
- 표본 수가 적은 공휴일이나 생산전환 조건은 단정하지 않습니다.
- 설비 상태와 제품 종류가 제공되지 않아 생산량만으로 모든 원인을 설명할 수 없습니다.
- 대표 사례는 오류가 발생하는 모습을 설명하기 위한 사례이며 전체 성능을 대표하지 않습니다.
- 최종 경보와 성능표는 176 기준을 유지합니다. 164.5~176은 현장 확인을 앞당기기 위한 주의 단계입니다.
- 주의 단계는 미탐지를 줄이는 대신 실제 피크가 아닌 시점에도 확인 요청이 늘어나는 운영상 선택입니다.
- 428건·171건은 매일 00시 예측 7,104칸의 오류분석 수치이고, 8~9월의 7,832건·1,460건은 모든 24시간 예측창을 사용한 별도 비교 수치입니다.

보고서는 다음 순서로 작성할 수 있습니다.

> 전체 오류 규모 → 수치예측이 잘 맞는 조건과 오차가 큰 조건 → 미탐지와 오경보 조건 → 주의 단계 추가 효과 → 대표 실패 사례 → 현장 적용 시 주의할 조건 → 4장의 피크 저감 조치로 연결
